# Kaffeequalität: EDA und Modellierung (Coffee Quality: EDA and Modelling)

Ein durchgerechnetes Beispiel für den Machine-Learning-Workflow (**workflow**, Arbeitsablauf). Wir nutzen den Datensatz zur Qualität von Arabica-Kaffee, um aus den gemessenen Eigenschaften jedes Kaffees seine Gesamtpunktzahl in der Verkostung vorherzusagen, und prüfen dann, wie gut das Modell ist. Nutze dieses Notebook als Vorlage für den Aufbau deines eigenen Projekts.

## Der Machine-Learning-Workflow (The Machine Learning Workflow)

### Das Geschäftsziel festlegen (Define the Business Goal)

Der **Stakeholder** (Auftraggeber) ist ein **Einkäufer für Rohkaffee bei einem Kaffeeimporteur**. Jede Saison prüft er viele Arabica-Partien (**lots**) und möchte seine Zeit und sein Budget für die vielversprechendsten einsetzen.

Wir schätzen die Gesamtpunktzahl der Verkostung (**cupping score**) eines Kaffees aus Eigenschaften, die man *vor* einer vollständigen Verkostung kennt. So kann der Einkäufer eine Vorauswahl der Partien treffen, die er probieren will, Preisverhandlungen unterstützen und vermeiden, Verkostungen für Kaffees von geringer Qualität zu verschwenden.

Die Punktzahl hängt von vielen Eigenschaften gleichzeitig ab. Darum liefert ein Modell eine schnellere und einheitlichere Schätzung als eine Faustregel von Hand.

Die Schätzung sollte so genau sein, dass sie in der Praxis nützt, zum Beispiel höchstens ein paar Punkte daneben auf der Skala von 0 bis 100.

In der Sprache der Data Science ist das ein **Regressionsproblem** (**regression**): Die Zielvariable `total_cup_points` ist eine stetige Zahl, und wir messen den Fehler mit dem **RMSE** (**root mean squared error**, Wurzel des mittleren quadratischen Fehlers). Er hat dieselbe Einheit wie die Punktzahl (Punkte).

### Die Daten beschaffen (Get the Data)

Wir laden den bereinigten Arabica-Datensatz. Er enthält sowohl die Eigenschaften der Kaffees als auch die Gesamtpunktzahl der Verkostung (`Total.Cup.Points`), die wir als Zielvariable (**target**) verwenden.

In [ ]:
import pandas as pd  # pandas für Tabellen laden # Kap. 15

# Arabica-Kaffeedaten (bereinigt)
URL = "https://raw.githubusercontent.com/jldbc/coffee-quality-database/master/data/arabica_data_cleaned.csv"  # Internetadresse der CSV-Datei # Kap. 4
coffee = pd.read_csv(URL)  # CSV direkt aus dem Internet als Tabelle einlesen # Kap. 15

Schauen wir uns die Daten kurz an, um zu sehen, womit wir arbeiten.

In [ ]:
coffee.head()  # erste 5 Zeilen anzeigen # Kap. 15

### Ein erster Blick auf die Daten (First Look at the Data)

Eine schnelle Prüfung des Aufbaus zeigt die Datentypen der Spalten und wo Werte fehlen.

In [ ]:
coffee.info()  # Spalten, Datentypen und fehlende Werte anzeigen # Kap. 15

Der Datensatz enthält 1311 Arabica-Kaffees mit 44 Spalten: eine Mischung aus Text (Herkunft, Farm, Verarbeitungsmethode) und Zahlen (Verkostungspunkte, Höhenlage, Anzahl der Mängel). Mehrere Spalten haben fehlende Werte, und die Höhenlage enthält auch einige unmögliche Einträge. Das alles bereinigen wir als Nächstes.

## Die Daten bereinigen (Clean the Data)

Alle diese Schritte sind **sicher vor Datenlecks** (**leakage-safe**), darum machen wir sie vor dem Aufteilen: Spaltennamen aufräumen, unbrauchbare Spalten und Zeilen entfernen, offensichtliche Eingabefehler korrigieren und Merkmale entfernen, die die Antwort verraten würden. Nur einen Schritt heben wir für nach dem Aufteilen auf: das Füllen fehlender Werte (**imputation**), weil dessen Füllwerte aus den Daten gelernt werden.

In [ ]:
# Spaltennamen aufräumen: Kleinbuchstaben, Unterstriche statt Punkten und Leerzeichen
coffee.columns = (  # neue Spaltennamen zuweisen, Klammer auf # Kap. 17
    coffee.columns.str.strip()  # Leerzeichen am Anfang und Ende entfernen # Kap. 5
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)  # alles, was kein Buchstabe oder keine Ziffer ist, durch _ ersetzen # neu
    .str.strip("_")  # Unterstriche am Anfang und Ende entfernen # Kap. 5
    .str.lower()  # alles in Kleinbuchstaben # Kap. 5
)  # Klammer schließen # Kap. 17
coffee.columns.tolist()  # neue Spaltennamen als Liste anzeigen # neu

Die unordentlichen Namen (zum Beispiel `Category.One.Defects` und `Total.Cup.Points`) werden zu sauberem **snake_case** (klein geschrieben, mit Unterstrichen). Das ist nur Umbenennen und darum immer sicher.

In [ ]:
# Spalten entfernen, bei denen mehr als die Hälfte der Werte fehlt
missing_fraction = coffee.isna().mean().sort_values(ascending=False)  # Anteil fehlender Werte pro Spalte, größter zuerst # Kap. 17
high_missing = missing_fraction[missing_fraction > 0.5].index  # Namen der Spalten mit über 50 % Lücken # Kap. 16
print("Dropping columns missing more than 50% of values:", list(high_missing))  # diese Spalten ausgeben # Kap. 4
coffee = coffee.drop(columns=high_missing)  # diese Spalten entfernen # Kap. 17

Nur bei `lot_number` fehlt mehr als die Hälfte der Werte, darum entfernen wir diese Spalte.

In [ ]:
# Eingabefehler bei der Höhenlage korrigieren: 10000 Meter oder mehr sind unmöglich
# (so hoch wächst kein Kaffee), es ist also fast sicher ein Tippfehler um eine Zehnerpotenz.
# Jeden solchen Wert so lange durch 10 teilen, bis er in einem glaubhaften Bereich liegt.
altitude_cols = ["altitude_low_meters", "altitude_high_meters", "altitude_mean_meters"]  # Liste der drei Höhen-Spalten # Kap. 8
print("Altitude values of 10000+ before fix:", int((coffee[altitude_cols] >= 10000).sum().sum()))  # wie viele Werte ab 10000 es gibt # Kap. 16
for col in altitude_cols:  # jede Höhen-Spalte durchgehen # Kap. 7
    while (coffee[col] >= 10000).any():  # solange noch ein Wert ab 10000 vorkommt ... # Kap. 7
        coffee[col] = coffee[col].where(coffee[col] < 10000, coffee[col] / 10)  # ... zu große Werte durch 10 teilen, andere behalten # neu
print("Highest altitude after fix:", round(coffee["altitude_mean_meters"].max(), 1))  # höchste Höhe nach der Korrektur # Kap. 4

Die Höhen-Spalten enthalten unmögliche Werte wie 190164 Meter. Das sieht nach Tippfehlern um eine Zehnerpotenz aus, darum teilen wir jede Höhe ab 10000 so lange durch 10, bis sie glaubhaft ist. Das ist eine feste Regel pro Zeile und darum sicher vor Datenlecks. Sie korrigiert 12 Werte in den drei Höhen-Spalten, und die größte Höhe sinkt auf etwa 4287 Meter.

In [ ]:
# Ein Kaffee hat die Gesamtpunktzahl 0, das ist unmöglich. Diese Zeile entfernen.
print("Rows with a total score of 0:", int((coffee["total_cup_points"] == 0).sum()))  # wie viele Zeilen 0 Punkte haben # Kap. 16
coffee = coffee[coffee["total_cup_points"] > 0]  # nur Zeilen mit mehr als 0 Punkten behalten # Kap. 16

Alle anderen Kaffees haben zwischen etwa 59 und 91 Punkten, eine Punktzahl von 0 ist also eindeutig ein Eingabefehler. Wir entfernen diese eine Zeile, übrig bleiben 1310 Kaffees.

In [ ]:
# Fachwissen: Die Gesamtpunktzahl ist die SUMME der einzelnen Verkostungspunkte.
# Das bestätigen wir und behandeln diese Einzelwerte dann als Datenleck.
component_scores = [  # Liste der 10 Einzelwertungen # Kap. 8
    "aroma", "flavor", "aftertaste", "acidity", "body",  # Aroma, Geschmack, Nachgeschmack, Säure, Körper # Kap. 8
    "balance", "uniformity", "clean_cup", "sweetness", "cupper_points",  # Balance, Gleichmäßigkeit, Reinheit, Süße, Gesamteindruck # Kap. 8
]  # Liste schließen # Kap. 8
sum_corr = coffee[component_scores].sum(axis=1).corr(coffee["total_cup_points"])  # Summe pro Zeile mit der Gesamtpunktzahl korrelieren # neu
print("Correlation of the component sum with the total:", round(sum_corr, 3))  # Korrelation ausgeben # Kap. 4

Die Einzelwertungen ergeben zusammen die Gesamtpunktzahl (Korrelation 1,00). Damit sind sie ein **Datenleck der Zielvariable** (**target leakage**): Gibt man sie dem Modell, bekommt es die Antwort einfach in zehn Teile zerlegt. Außerdem passen sie nicht zu unserem Ziel, denn der Einkäufer möchte eine Schätzung *vor* der Verkostung, wenn es diese Punkte noch gar nicht gibt.

Darum entfernen wir die Einzelwertungen jetzt und sagen die Qualität stattdessen aus wirklich unabhängigen Eigenschaften vorher.

In [ ]:
# Die Zielvariable ist die Gesamtpunktzahl der Verkostung
y = coffee["total_cup_points"]  # Zielspalte # Kap. 16

# Features: Zahlen-Spalten ohne Zeilennummer, Zielvariable und die verräterischen Einzelwertungen
X = coffee.select_dtypes("number").drop(  # nur Zahlen-Spalten nehmen, dann entfernen ... # neu
    columns=["unnamed_0", "total_cup_points"] + component_scores  # ... Zeilennummer, Ziel und die 10 Einzelwertungen # Kap. 17
)  # Klammer schließen # Kap. 17
X.columns.tolist()  # übrige Feature-Namen anzeigen # neu

Übrig bleiben acht unabhängige Eigenschaften: die Anzahl der Säcke, die Feuchtigkeit, die zwei Arten von Mängeln, die „quakers“ (unreife, helle Bohnen) und die drei Höhen-Spalten. Das alles kann ein Einkäufer über eine Partie wissen, ohne sie zu verkosten.

## Aufteilen in Training und Test (Train-Test Split)

Wir halten einen Teil der Daten zurück, damit wir das Modell an Kaffees bewerten können, die es nie gesehen hat. Ab hier wird alles, was wir *fitten* (lernen lassen), zum Beispiel Füllwerte für Lücken, nur auf den Trainingsdaten gelernt.

Wir nutzen eine einzige Aufteilung (80 % Training, 20 % Test). Ein größeres Projekt würde zusätzlich einen Validierungsdatensatz behalten oder Kreuzvalidierung nutzen, um das Modell abzustimmen, ohne den Testdatensatz anzufassen.

In [ ]:
from sklearn.model_selection import train_test_split  # Werkzeug zum Aufteilen laden # neu

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)  # 80 % Training, 20 % Test, fester Zufallswert # neu

Die 1310 Kaffees werden in 1048 für das Training und 262 für den Test aufgeteilt.

## Die Trainingsdaten erkunden (Explore the Training Data)

Jetzt erkunden wir den **Trainingsdatensatz**, um die Features zu verstehen, bevor wir irgendetwas fitten.

In [ ]:
X_train.info()  # Spalten, Datentypen und Lücken im Training # Kap. 15

Es gibt acht numerische Features. Die meisten sind vollständig, aber bei `quakers` fehlt ein Wert, und die drei Höhen-Spalten fehlen bei 187 der 1048 Trainingskaffees. Diese Lücken füllen wir nach dem Aufteilen.

In [ ]:
# Wie stark hängt jedes übrige Feature mit der Zielvariable zusammen?
X_train.corrwith(y_train).sort_values(ascending=False)  # Korrelation jeder Spalte mit y, größte zuerst # neu

Keine der unabhängigen Eigenschaften hängt stark mit der Punktzahl zusammen. Das deutlichste Signal ist `category_two_defects` (etwa −0,28: mehr Mängel, weniger Punkte), danach folgen Feuchtigkeit und Höhenlage, alle schwach.

## Feature Engineering (Merkmale aufbereiten)

Wir füllen jeden fehlenden Wert (`quakers` und die drei Höhen-Spalten) mit dem Mittelwert der Spalte aus den **Trainingsdaten** und verwenden genau diese Werte später auch für den Testdatensatz.

In [ ]:
# Die Füllwerte nur aus den Trainingsdaten lernen
train_means = X_train.mean()  # Mittelwert jeder Spalte im Training # Kap. 15
X_train = X_train.fillna(train_means)  # Lücken im Training mit diesen Mittelwerten füllen # Kap. 17
train_means  # Mittelwerte anzeigen # Kap. 15

`train_means` enthält einen Wert pro Spalte. Genau diese Werte verwenden wir später bei der Bewertung auch für den Testdatensatz. Das hält die Bewertung ehrlich.

## Baseline-Modell (Baseline Model)

Bevor wir ein Modell trainieren, legen wir eine **Baseline** fest: die einfachste Regel, die eine Vorhersage macht.

Bei einer Regression wie unserer ist eine übliche Faustregel, die Features zu ignorieren und immer die **durchschnittliche Punktzahl aus dem Training** vorherzusagen. Wenn unser Modell das nicht schlägt, hat es nichts Nützliches gelernt.

In [ ]:
# Faustregel-Baseline: Features ignorieren, immer den Trainings-Mittelwert vorhersagen
from sklearn.metrics import root_mean_squared_error  # RMSE-Funktion laden # neu

baseline_prediction = y_train.mean()  # Mittelwert der Punktzahl im Training # Kap. 15
baseline_rmse = root_mean_squared_error(y_test, [baseline_prediction] * len(y_test))  # RMSE, wenn man für jeden Testkaffee diesen Mittelwert sagt # Kap. 8

print(f"Baseline always predicts: {baseline_prediction:.2f}")  # was die Baseline immer sagt # Kap. 5
print(f"Baseline test RMSE: {baseline_rmse:.2f}")  # Fehler der Baseline auf dem Test # Kap. 5

Die Baseline sagt immer etwa 82,2 vorher (den Trainings-Mittelwert) und erreicht auf dem Test einen RMSE von etwa 2,66. Sie liegt im Schnitt also etwa 2 bis 3 Punkte daneben. Diese Zahl muss unser Modell schlagen.

## Das Modell trainieren (Train the Model)

Wir trainieren eine lineare Regression: schnell, leicht zu lesen und ein sinnvoller Schritt über die Baseline hinaus. Zuerst legen wir den Testdatensatz beiseite und speichern eine Kopie, dann fitten wir das Modell und prüfen den Fehler auf den Trainingsdaten.

In [ ]:
# Testdatensatz beiseitelegen und eine Kopie speichern. Bis zur abschließenden Bewertung fassen wir ihn nicht an.
X_test.to_csv("data/X_test.csv")  # Test-Features als CSV im Ordner data speichern # Kap. 15
y_test.to_csv("data/y_test.csv")  # Test-Zielwerte als CSV speichern # Kap. 15

In [ ]:
from sklearn.linear_model import LinearRegression  # lineare Regression laden # neu
reg = LinearRegression().fit(X_train, y_train)  # Modell anlegen und auf den Trainingsdaten trainieren # neu

In [ ]:
y_train_pred = reg.predict(X_train)  # Vorhersagen für die Trainingsdaten # neu
rmse = root_mean_squared_error(y_train, y_train_pred)  # Fehler auf den Trainingsdaten # neu
print(rmse)  # ausgeben # Kap. 4

Der RMSE auf den Trainingsdaten liegt bei etwa 2,54 Punkten, also nahe an der Baseline. Das Modell lernt die Daten nicht auswendig. Das überrascht nicht, weil die Features nur schwach mit der Punktzahl zusammenhängen.

## Das Modell bewerten (Evaluate the Model)

Wir wenden denselben gelernten Schritt auf den Testdatensatz an: das Füllen mit den Mittelwerten aus dem **Training** (nie neu aus den Testdaten berechnet). Die Bereinigung und die Korrektur der Höhenlage sind schon vor dem Aufteilen passiert, darum hat der Testdatensatz sie auch. Dann bewerten wir das Modell an diesen ungesehenen Kaffees.

In [ ]:
# Den gelernten Schritt (Füllen der Lücken) auf den Test anwenden, mit den Trainings-Mittelwerten
X_test = X_test.fillna(train_means)  # Lücken im Test mit den Mittelwerten aus dem Training füllen # Kap. 17

In [ ]:
y_test_pred = reg.predict(X_test)  # Vorhersagen für die Testdaten # neu
rmse = root_mean_squared_error(y_test, y_test_pred)  # Fehler auf den Testdaten # neu
print(rmse)  # ausgeben # Kap. 4

Der RMSE auf dem Test liegt bei etwa 2,46, also knapp unter den 2,66 der Baseline. Das Modell ist nur ein wenig besser als immer den Durchschnitt zu raten: Die Eigenschaften, die ein Einkäufer vor der Verkostung kennen kann (Höhenlage, Mängel, Feuchtigkeit, Anzahl der Säcke), sagen nur wenig über die endgültige Punktzahl aus.

Die nächsten Schritte im Workflow wären: reichhaltigere Features oder Modelle ausprobieren, das beste mit Validierungsdaten oder Kreuzvalidierung auswählen, Hyperparameter abstimmen, eine Fehleranalyse machen, einen finalen Test-Score berichten und das Modell dann einsetzen und überwachen.

In [ ]:
# Das beste Modell für später in einer Datei speichern
import joblib  # Modul zum Speichern und Laden von Modellen # neu
joblib.dump(reg, "models/linear_regression_model.joblib")  # Modell im Ordner models speichern # neu

In [ ]:
# Mit dem Modell die Punktzahl eines neuen Kaffees mit diesen Eigenschaften vorhersagen:

new_coffee = pd.DataFrame({  # Tabelle mit einem neuen Kaffee bauen, Klammer auf # Kap. 15

    "number_of_bags" :[1] ,  # 1 Sack # Kap. 10
    "moisture": [10.5] ,  # Feuchtigkeit 10,5 # Kap. 10
    "category_one_defects": [0],  # keine schweren Mängel # Kap. 10
    "quakers": [0],  # keine unreifen Bohnen # Kap. 10
    "category_two_defects": [1],  # 1 leichter Mangel # Kap. 10
    "altitude_low_meters": [1200],  # niedrigste Höhe 1200 m # Kap. 10
    "altitude_high_meters": [1500],  # höchste Höhe 1500 m # Kap. 10
    "altitude_mean_meters": [1350]  # mittlere Höhe 1350 m # Kap. 10
})  # Klammer schließen # Kap. 15
new_coffee = new_coffee.fillna(train_means)  # dieselbe Füllung wie bei den Trainingsdaten anwenden # Kap. 17

In [ ]:
X_train.columns  # Spaltennamen im Training anzeigen, zur Kontrolle # Kap. 15

In [ ]:
# Das Modell aus der Datei laden und damit die Punktzahl des neuen Kaffees vorhersagen
loaded_model = joblib.load("models/linear_regression_model.joblib")  # gespeichertes Modell laden # neu
predicted_score = loaded_model.predict(new_coffee)[0]  # Vorhersage, erste (einzige) Zahl nehmen # Kap. 8
print(f"Predicted cupping score for the new coffee: {predicted_score:.2f}")  # Ergebnis ausgeben # Kap. 5

## Quellen & Weiterlesen (References & Further Reading)

- [**LinearRegression (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html): das in diesem Notebook trainierte Modell.
- [**train_test_split (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html): wie der zurückgehaltene Testdatensatz erstellt wird.
- [**root_mean_squared_error (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): die Metrik, mit der Modell und Baseline bewertet werden.
- [**Common pitfalls and recommended practices (scikit-learn)**](https://scikit-learn.org/stable/common_pitfalls.html): wie Datenlecks entstehen und wie man sie vermeidet.
- [**Coffee Quality Database**](https://github.com/jldbc/coffee-quality-database): die Quelle des hier verwendeten Datensatzes.